# RMSNorm向量算子实现与流水优化

RMSNorm通过一行输入元素的均方根完成归一化，并使用缩放权重对各特征位置进行逐元素调整。其计算同时包含Vector逐元素运算、行内归约、全局内存（Global Memory，GM）数据搬运和片上流水组织，适合用于观察算子融合、tile划分及双缓冲对性能的影响。

本实验学习大纲如下：

1. 实验概述：介绍实验目标、前置知识和实验要点；
2. 环境准备：创建`Source/04.03`实验工程并加载CANN环境；
3. 问题分析：理解RMSNorm公式、数据布局、tile划分和整行归约依赖；
4. 核函数开发：实现不同实验方案的核函数开发；
5. 结果验证与性能分析：完成Host侧数据准备、Device内存管理、Kernel启动、误差校验和性能统计，并比较不同输入规模和tileLen的影响；
6. 实验总结：归纳算子融合、片上中间量复用和双缓冲流水组织的实现过程与性能影响。


---
## 1. 实验概述

RMSNorm是深度学习模型中常用的归一化操作，通过输入元素的均方根对特征数据进行归一化，并使用缩放权重对归一化结果进行逐元素调整。其计算过程包含逐元素平方、行内归约、均方值计算、倒数平方根计算和逐元素乘法等操作，适合用于说明Vector计算、tile划分、算子融合和片上流水之间的关系。本实验围绕行级RMSNorm向量算子展开，设置基线方案、单Kernel连续计算实验方案和双缓冲流水执行实验方案三种对比实验方案。其中，基线方案使用两个Kernel，先完成逐元素平方并将中间结果写回全局内存（Global Memory，GM），再读取平方结果完成行内归约、归一化因子计算和权重缩放；单Kernel连续计算实验方案在一个Kernel内完成全部计算，减少中间结果对GM的写回和读取；双缓冲流水实验方案保持与单Kernel实验方案相同的计算流程，通过双缓冲，使相邻tile的数据搬入、Vector计算和结果写回能够按阶段推进。通过比较三种实验方案的计算结果、实验误差、Kernel执行时间、算子调用耗时和估算有效带宽，分析算子融合以及双缓冲流水组织对RMSNorm向量算子性能的影响。


### 1.1 实验目标

完成本实验后应达到以下目标：

1. 理解RMSNorm的数据组织关系和行级归一化流程。掌握输入张量与输出张量之间的关系，理解平方计算、行内归约、倒数平方根计算等核心步骤。
2. 掌握Vector算子的基本开发流程和基于Ascend C基础接口实现RMSNorm向量算子的方法。理解数据从Global Memory搬入LocalTensor，调用相关Vector接口完成平方计算、行内归约和缩放计算，再将结果写回Global Memory的基本过程。
3. 具备正确性验证和性能分析能力。结合运行日志、性能指标和Profiling工具，对比不同实验方案的计算误差、Kernel执行时间、算子调用耗时和估算有效带宽，分析算子融合和双缓冲流水对性能的影响，并理解不同输入规模下流水管理开销与优化收益之间的关系。


### 1.2 前置知识

本实验要求提前具备以下基础：

1. 二维张量与连续存储基础：理解输入张量X可以看作M行N列的二维数组，明确二维张量在计算机中通常以连续的一维数组保存。应掌握行列下标与线性内存位置之间的对应关系，并理解输入和输出之间的数据组织关系。
2. RMSNorm计算基础：理解RMSNorm的数学含义和行级计算过程，明确每一行需要依次完成平方计算、行内平方和归约、均方值计算和倒数平方根计算，再将输入元素与归一化因子及缩放权重逐元素相乘，得到该行的输出结果。应理解同一行中的所有输出元素共享相同的归一化因子，并认识到必须完成整行归约后才能计算该行的输出。
3. tile划分与行内归约基础：理解tile化处理的基本思想，即将一行较长的数据划分为多个较小的数据块，使数据搬入、向量计算和结果写回能够围绕tile进行组织。进一步理解每个tile可以先计算局部平方和，再将多个局部结果累加为整行平方和，并理解tile长度、每行tile数量和补齐行宽对数据访问及流水性能的影响。
4. Ascend C开发基础：理解Host侧和Device侧的基本分工。Host侧负责输入准备、数据补齐、Device内存申请、Kernel启动、运行计时和结果校验；Device侧执行Kernel，并完成数据搬运、向量计算和行内归约。
5. 静态Tensor与双缓冲流水基础：理解静态Tensor编程需要确定数据块边界和LocalTensor的静态容量，并根据数据搬入、计算和写回之间的依赖关系管理同步。理解数据通常先从Global Memory搬入Local Memory，再在片上完成计算，最后写回Global Memory。理解双缓冲通过两组片上缓冲交替工作，使相邻tile的数据搬入、计算和写回能够按流水阶段推进，同时也会引入队列管理、流水启动和排空开销。


### 1.3 实验要点

实验中应重点关注以下内容：

1. RMSNorm数据组织：按照连续内存方式保存输入张量、缩放权重和输出张量，明确每行数据在特征维度N上独立完成归一化，并理解Device实现中数据分块和行宽补齐对存储位置及有效计算范围的影响，保证Host侧与Device侧的数据组织一致。
2. 基线方案实现：按照RMSNorm的计算流程分阶段实现平方计算、行内归约、归一化因子计算和权重缩放与结果写回，保留必要的中间结果。该方案计算过程直观，便于验证功能正确性，并作为后续实验方案的性能对照基准。
3. 单Kernel连续计算：在一个Kernel内完成平方计算、行内归约、倒数平方根计算和权重缩放与结果写回，复用片上中间结果，减少平方结果、均值结果和归一化中间量对Global Memory的反复读写。
4. 流水执行组织：在长行输入场景下按照tile组织数据搬入和片上计算，并使用双缓冲协调数据搬入、片上计算和结果写回。先完成一行数据的平方和累加，再计算归一化因子，随后对该行数据进行权重缩放与结果写回。
5. 结果正确性验证：使用Host侧参考结果对Device侧输出进行逐项比较，结合最大绝对误差、最大相对误差、错误元素数量和运行状态，判断三个实验方案的计算结果是否正确可信。
6. 性能结果分析：分记录基础实验方案、单Kernel连续实验方案和双缓冲流水实验方案的Kernel执行时间、算子调用耗时、估算有效带宽和访存量，结合Profiling结果分析算子融合、片上中间量复用和流水执行对性能的影响，并解释不同输入规模下流水管理开销与优化收益之间的关系。


---
## 2. 环境准备

首先创建实验所需目录，并尝试加载Ascend CANN环境变量。

- `Source/04.03/include`：公共参数、性能指标和Host参考校验工具；
- `Source/04.03/ascend_ops/op_kernel`：三个RMSNorm Device实现；
- `Source/04.03/ascend_ops/host_launch`：ACL运行时管理和Kernel启动代码；
- `Source/04.03/scripts`：构建、运行和Profiling脚本；
- `Source/04.03/results`：保存三种实验方案的运行日志与汇总结果；
- `Source/04.03/CMakeLists.txt`：配置Ascend C核函数及NPU Host程序。


In [ ]:
from pathlib import Path
import os
import shlex
import shutil
import subprocess

PROJECT_ROOT = Path("Source/04.03")
for directory in [
    "include",
    "scripts",
    "results",
    "ascend_ops/op_kernel",
    "ascend_ops/host_launch",
]:
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)

def find_cann_root(env_script):
    for parent in [env_script.parent, *env_script.parents]:
        cmake_candidates = [
            parent / "tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
        ]
        if any(path.is_file() for path in cmake_candidates):
            return parent
    return None

candidate_scripts = []
for item in [
    os.environ.get("ASCEND_TOOLKIT_HOME"),
    os.environ.get("ASCEND_INSTALL_PATH"),
    "/usr/local/Ascend/ascend-toolkit/latest",
]:
    if item:
        candidate_scripts.append(Path(item) / "set_env.sh")

for root in [Path("/usr/local/Ascend"), Path("/opt/Ascend"),
             Path.home() / "Ascend", Path("/workspace/Ascend")]:
    if root.exists():
        candidate_scripts.extend(sorted(root.glob("**/set_env.sh"), reverse=True))

set_env = None
install_root = None
for script in candidate_scripts:
    if not script.is_file():
        continue
    root = find_cann_root(script)
    if root is not None:
        set_env = script
        install_root = root
        break

if set_env is not None and shutil.which("bash"):
    command = f"source {shlex.quote(str(set_env))} && env"
    loaded_env = subprocess.check_output(["bash", "-lc", command], text=True)
    for line in loaded_env.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
    os.environ["ASCEND_INSTALL_PATH"] = str(install_root)
    os.environ["ASCEND_CANN_PACKAGE_PATH"] = str(install_root)
    print("Ascend environment loaded from:", set_env)
    print("ASCEND_INSTALL_PATH:", install_root)
else:
    print("Ascend set_env.sh was not found. Code generation can continue; build/run needs CANN.")

print("Experiment directory:", PROJECT_ROOT.resolve())
print("cmake:", shutil.which("cmake") or "not found")
print("C++ compiler:", shutil.which("g++") or shutil.which("c++") or "not found")


---
## 3. 问题分析

### 3.1 RMSNorm计算目标

设输入张量为$X\in\mathbb{R}^{M\times N}$，缩放权重为$\gamma\in\mathbb{R}^{N}$。第$i$行的归一化因子为：

$$
r_i=\frac{1}{\sqrt{\frac{1}{N}\sum_{j=0}^{N-1}x_{i,j}^{2}+\epsilon}}
$$

输出为：

$$
y_{i,j}=x_{i,j}\cdot r_i\cdot\gamma_j
$$

$\epsilon$是加入均方值的数值稳定项，本实验取$10^{-5}$。同一行的所有输出共享$r_i$，因此必须先完成整行平方和归约，才能计算该行输出。


### 3.2 数据布局、补齐与tile划分

输入`X[M,N]`和输出`Y[M,N]`采用行优先连续存储，线性位置为`row * paddedN + col`。缩放权重`gamma[N]`由所有行共享，但本实验不跨行缓存gamma，确保单Kernel连续计算实验方案和双缓冲流水实验方案除流水组织外保持相同的数据访问方式。

为了使用固定长度整块搬运，Host将行宽按`tileLen`向上补齐：

$$
paddedN=\left\lceil\frac{N}{tileLen}\right\rceil tileLen
$$

每行tile数量为`paddedN/tileLen`。本次实验默认`N=1024`、`tileLen=256`，因此`paddedN=1024`且每行包含4个tile。


### 3.3 三种实验方案

| 实验方案 | Kernel数量 | 计算组织 | 主要GM访问 |
|---|---:|---|---|
| 基线实验方案（命令参数`basic`） | 2 | 先平方并写回中间结果，再归约和计算输出 | 读X、写/读平方结果、再读X和gamma、写Y |
| 单Kernel连续计算实验方案（命令参数`fused`） | 1 | 一个Kernel内完成整行归约和输出计算 | 两遍读取X、读取gamma、写Y |
| 双缓冲流水实验方案（命令参数`pipeline`） | 1 | 与单Kernel连续计算实验方案相同的两遍计算，使用深度为2的队列组织tile | 与单Kernel连续计算实验方案一致 |

因此，基线实验方案到单Kernel连续计算实验方案的变化同时体现Kernel融合和中间结果GM访问减少；单Kernel连续计算实验方案到双缓冲流水实验方案的核心变量是双缓冲队列带来的阶段化流水。


### 3.4 RMSNorm依赖与双缓冲

RMSNorm不能在尚未得到整行平方和时直接输出某个tile。双缓冲流水实验方案仍分为两遍：

1. 第一遍按tile搬入X，计算各tile局部平方和并累加为整行平方和；
2. 根据整行平方和计算均方值，在均方值上加上数值稳定项epsilon，再计算平方根的倒数，得到该行所有元素共享的归一化因子`invRms`，计算公式为：

   $$
   \mathrm{meanSquare}_i=\frac{\mathrm{squareSum}_i}{N},\qquad
   \mathrm{invRms}_i=\frac{1}{\sqrt{\mathrm{meanSquare}_i+\epsilon}}
   =\frac{1}{\sqrt{\frac{\mathrm{squareSum}_i}{N}+\epsilon}}
   $$

   其中，$\mathrm{squareSum}_i$表示第$i$行所有元素的平方和，N表示每行参与归一化计算的元素数量。
3. 第二遍重新按tile搬入X和gamma，完成逐元素缩放与写回。

双缓冲发生在每一遍内部的相邻tile之间。`TQue<...,2>`提供两个交替使用的片上缓冲，使MTE2数据搬入、Vector计算和MTE3结果写回具备重叠条件。流水同时会增加预取、队列管理、启动和排空开销，因此小规模输入下不一定更快。


### 3.5 公共参数与性能指标

公共头文件统一定义实验方案枚举、输入参数、计时结果、误差指标、边界检查和有效带宽估算。估算搬运量按照每个补齐元素涉及的FP32读写次数计算：基线实验方案（`basic`）按6个float估算，单Kernel连续计算实验方案（`fused`）和双缓冲流水实验方案（`pipeline`）按4个float估算。

有效带宽为：

$$
BW_{effective}=\frac{estimatedMovedBytes}{total\_us\times10^{-6}}\div10^9
$$

它用于当前三种方案的相对比较，不等同于硬件物理带宽上限。


In [ ]:
%%writefile Source/04.03/include/rmsnorm_common.h

#pragma once

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <iomanip>
#include <iostream>
#include <limits>
#include <stdexcept>
#include <string>
#include <vector>

namespace rmsnorm {

enum class Version {
    Basic,
    Fused,
    Pipeline
};

struct Config {
    uint32_t m = 256;
    uint32_t n = 1024;
    uint32_t tile_len = 256;
    uint32_t block_dim = 8;
    uint32_t seed = 1234;
    float epsilon = 1.0e-5f;
    Version version = Version::Fused;
    bool sweep = false;
    bool print_output = false;
};

struct TimingUs {
    double kernel_us = 0.0;
    double total_us = 0.0;
};

struct Metrics {
    double max_abs_error = 0.0;
    double max_rel_error = 0.0;
    uint64_t mismatch_count = 0;
};

class Timer {
public:
    Timer() : start_(std::chrono::high_resolution_clock::now()) {}
    double elapsed_us() const {
        const auto end = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::micro>(end - start_).count();
    }
private:
    std::chrono::high_resolution_clock::time_point start_;
};

inline const char* version_name(Version version) {
    switch (version) {
        case Version::Basic: return "basic";
        case Version::Fused: return "fused";
        case Version::Pipeline: return "pipeline";
    }
    return "unknown";
}

inline Version parse_version(const std::string& value) {
    if (value == "basic") {
        return Version::Basic;
    }
    if (value == "fused") {
        return Version::Fused;
    }
    if (value == "pipeline") {
        return Version::Pipeline;
    }
    throw std::invalid_argument("--version must be one of: basic, fused, pipeline");
}

inline uint64_t round_up(uint64_t n, uint64_t align) {
    if (align == 0) {
        throw std::invalid_argument("align must be > 0");
    }
    const uint64_t rem = n % align;
    if (rem == 0) {
        return n;
    }
    const uint64_t add = align - rem;
    if (n > std::numeric_limits<uint64_t>::max() - add) {
        throw std::overflow_error("round_up overflow");
    }
    return n + add;
}

inline uint32_t ceil_div_u32(uint64_t n, uint64_t d) {
    if (d == 0) {
        throw std::invalid_argument("divisor must be > 0");
    }
    const uint64_t q = (n / d) + ((n % d) != 0 ? 1ull : 0ull);
    if (q > static_cast<uint64_t>(std::numeric_limits<uint32_t>::max())) {
        throw std::overflow_error("ceil_div_u32 result exceeds uint32_t range");
    }
    return static_cast<uint32_t>(q);
}

inline uint64_t checked_mul_u64(uint64_t a, uint64_t b, const char* name) {
    if (a != 0 && b > std::numeric_limits<uint64_t>::max() / a) {
        throw std::overflow_error(std::string(name) + " overflows uint64_t");
    }
    return a * b;
}

inline size_t checked_float_bytes(uint64_t count) {
    if (count > static_cast<uint64_t>(std::numeric_limits<size_t>::max() / sizeof(float))) {
        throw std::overflow_error("float byte size overflows size_t");
    }
    return static_cast<size_t>(count) * sizeof(float);
}

inline void check_config(uint32_t m, uint32_t n, uint32_t tile_len,
                         uint32_t dtype_size = sizeof(float)) {
    if (m == 0) {
        throw std::invalid_argument("m must be > 0");
    }
    if (n == 0) {
        throw std::invalid_argument("n must be > 0");
    }
    if (tile_len == 0) {
        throw std::invalid_argument("tile_len must be > 0");
    }
    if ((static_cast<uint64_t>(tile_len) * dtype_size) % 32 != 0) {
        throw std::invalid_argument("tile_len * sizeof(T) must be a multiple of 32 bytes");
    }
}

inline uint64_t valid_element_count(uint32_t m, uint32_t n) {
    return checked_mul_u64(m, n, "valid element count");
}

inline uint64_t padded_element_count(uint32_t m, uint32_t padded_n) {
    return checked_mul_u64(m, padded_n, "padded element count");
}

inline double estimated_moved_bytes(uint64_t padded_elements, Version version) {
    // basic: read X, write square, read square, read X, read gamma, write Y
    // fused: read X for reduction, read X again, read gamma for each row, write Y
    // pipeline: same GM traffic as fused; the intended difference is only the
    // double-buffer queue scheduling of CopyIn, vector Compute, and CopyOut.
    double floats_per_element = 4.0;
    if (version == Version::Basic) {
        floats_per_element = 6.0;
    }
    return static_cast<double>(padded_elements) * floats_per_element * sizeof(float);
}

inline double effective_gbps(uint64_t padded_elements, Version version, double total_us) {
    if (total_us <= 0.0) {
        return 0.0;
    }
    return estimated_moved_bytes(padded_elements, version) / (total_us * 1.0e-6) / 1.0e9;
}

inline void print_header() {
    std::cout << std::setw(8) << "M"
              << std::setw(8) << "N"
              << std::setw(10) << "paddedN"
              << std::setw(10) << "tileLen"
              << std::setw(10) << "tiles"
              << std::setw(8) << "cores"
              << std::setw(12) << "version"
              << std::setw(14) << "kernel_us"
              << std::setw(14) << "total_us"
              << std::setw(12) << "GB/s"
              << std::setw(14) << "max_abs"
              << std::setw(14) << "max_rel"
              << std::setw(10) << "errors"
              << std::setw(10) << "status"
              << "\n";
}

inline void print_result_row(uint32_t m,
                             uint32_t n,
                             uint32_t padded_n,
                             uint32_t tile_len,
                             uint32_t num_tiles_per_row,
                             uint32_t block_dim,
                             Version version,
                             const TimingUs& timing,
                             const Metrics& metrics) {
    const bool pass = metrics.mismatch_count == 0;
    const uint64_t padded_elements = padded_element_count(m, padded_n);

    std::cout << std::setw(8) << m
              << std::setw(8) << n
              << std::setw(10) << padded_n
              << std::setw(10) << tile_len
              << std::setw(10) << num_tiles_per_row
              << std::setw(8) << block_dim
              << std::setw(12) << version_name(version)
              << std::setw(14) << std::fixed << std::setprecision(2) << timing.kernel_us
              << std::setw(14) << std::fixed << std::setprecision(2) << timing.total_us
              << std::setw(12) << std::fixed << std::setprecision(3) << effective_gbps(padded_elements, version, timing.total_us)
              << std::setw(14) << std::scientific << std::setprecision(3) << metrics.max_abs_error
              << std::setw(14) << std::scientific << std::setprecision(3) << metrics.max_rel_error
              << std::setw(10) << std::defaultfloat << metrics.mismatch_count
              << std::setw(10) << (pass ? "PASS" : "FAIL")
              << "\n";
}

}


---
## 4. 核函数开发

### 4.1 公共Vector计算与多核按行划分

三种实验方案共用同一组行划分和Vector计算工具。`GetRowRange`是本实验封装的Device辅助函数，用于将连续行分配给不同AI Core，使各核心的写回区域互不重叠。

`SquareAndReduceVector`和`ScaleAndWeightVectorToOut`不是Ascend C内置接口，而是本实验根据RMSNorm计算流程封装的Device辅助函数：

- `SquareAndReduceVector`：先计算一个tile内各元素的平方，再对平方结果进行分层归约，返回该tile的局部平方和；
- `ScaleAndWeightVectorToOut`：先将输入tile与当前行共享的归一化因子`invRms`相乘，再与缩放权重gamma逐元素相乘，得到输出tile。

这两个辅助函数内部调用了Ascend C提供的Vector计算接口：

- `Mul`：Vector逐元素乘法接口。`Mul(xLocal, xLocal, xLocal, len)`用于计算逐元素平方，`Mul(yLocal, yLocal, gammaLocal, len)`用于将归一化结果与gamma逐元素相乘；
- `Muls`：Vector与标量相乘接口，用于计算`yLocal=xLocal*invRms`；
- `WholeReduceSum`：Vector归约求和接口。当一个tile包含较多元素时，先对若干组元素分别求和，再继续归约局部结果，直到得到一个tile的平方和。

`PIPE_V`只同步Vector流水内部存在依赖的指令，整段数据搬运边界仍由对应实验方案的同步或队列机制管理。


In [ ]:
%%writefile Source/04.03/ascend_ops/op_kernel/rmsnorm_static_tensor.cpp

#include "kernel_operator.h"

using namespace AscendC;

namespace {
constexpr uint32_t kStaticMaxTileLen = 8192;
constexpr uint32_t kPipelineMaxTileLen = 4096;
constexpr int32_t kPipelineBufferNum = 2;
constexpr uint32_t kFloatElementsPerRepeat = 64;
constexpr uint32_t kFloatElementsPerBlock = 8;
constexpr uint32_t kReduceFirstPassCount =
    (kStaticMaxTileLen + kFloatElementsPerRepeat - 1) / kFloatElementsPerRepeat;
constexpr uint32_t kReduceWorkspaceLen =
    ((kReduceFirstPassCount + kFloatElementsPerBlock - 1) / kFloatElementsPerBlock) *
    kFloatElementsPerBlock;

using PipelineInQueue = TQue<QuePosition::VECIN, kPipelineBufferNum>;
using PipelineOutQueue = TQue<QuePosition::VECOUT, kPipelineBufferNum>;

__aicore__ inline void GetRowRange(uint32_t rowCount,
                                   uint32_t launchBlockDim,
                                   uint32_t& beginRow,
                                   uint32_t& endRow) {
    const uint32_t coreNum = launchBlockDim == 0 ? 1 : launchBlockDim;
    const uint32_t coreId = GetBlockIdx();
    if (coreId >= coreNum) {
        beginRow = 0;
        endRow = 0;
        return;
    }
    const uint32_t rowsPerCore = (rowCount + coreNum - 1) / coreNum;
    beginRow = coreId * rowsPerCore;
    endRow = beginRow + rowsPerCore;
    if (endRow > rowCount) {
        endRow = rowCount;
    }
}

__aicore__ inline float InvSqrt(float value) {
    return 1.0f / sqrt(value);
}

__aicore__ inline float SumLocal(LocalTensor<float> tileLocal, uint32_t tileLen) {
    float sum = 0.0f;
    for (uint32_t i = 0; i < tileLen; ++i) {
        sum += tileLocal.GetValue(i);
    }
    return sum;
}

__aicore__ inline float SquareAndSumVector(LocalTensor<float> tileLocal, uint32_t len) {
    Mul(tileLocal, tileLocal, tileLocal, len);
    PipeBarrier<PIPE_ALL>();
    return SumLocal(tileLocal, len);
}

__aicore__ inline void ScaleAndWeightVector(LocalTensor<float> xLocal,
                                            LocalTensor<float> gammaLocal,
                                            float invRms,
                                           uint32_t len) {
    Muls(xLocal, xLocal, invRms, len);
    PipeBarrier<PIPE_ALL>();
    Mul(xLocal, xLocal, gammaLocal, len);
}

__aicore__ inline float SquareAndReduceVector(LocalTensor<float> xLocal,
                                              LocalTensor<float> reduceLocal,
                                              uint32_t len) {
    Mul(xLocal, xLocal, xLocal, len);
    PipeBarrier<PIPE_V>();

    uint32_t reduceCount = len;
    LocalTensor<float> reduceSrc = xLocal;
    while (reduceCount > 1) {
        const uint32_t fullRepeats = reduceCount / kFloatElementsPerRepeat;
        const uint32_t tailCount = reduceCount % kFloatElementsPerRepeat;
        if (fullRepeats > 0) {
            WholeReduceSum(reduceLocal,
                           reduceSrc,
                           static_cast<int32_t>(kFloatElementsPerRepeat),
                           static_cast<int32_t>(fullRepeats),
                           1,
                           1,
                           8);
        }
        if (tailCount > 0) {
            WholeReduceSum(reduceLocal[fullRepeats],
                           reduceSrc[fullRepeats * kFloatElementsPerRepeat],
                           static_cast<int32_t>(tailCount),
                           1,
                           1,
                           1,
                           8);
        }
        reduceCount = fullRepeats + (tailCount > 0 ? 1U : 0U);
        reduceSrc = reduceLocal;
        PipeBarrier<PIPE_V>();
    }

    PipeBarrier<PIPE_V>();
    return reduceLocal.GetValue(0);
}

__aicore__ inline void ScaleAndWeightVectorToOut(LocalTensor<float> yLocal,
                                                 LocalTensor<float> xLocal,
                                                 LocalTensor<float> gammaLocal,
                                                 float invRms,
                                                 uint32_t len) {
    Muls(yLocal, xLocal, invRms, len);
    PipeBarrier<PIPE_V>();
    Mul(yLocal, yLocal, gammaLocal, len);
}

}


### 4.2 基线实验方案：两个Kernel分阶段计算

`rmsnorm_basic_square`读取X、完成逐元素平方并把平方结果写入GM。`rmsnorm_basic_write`重新读取平方结果完成整行归约，再读取X和gamma计算输出。该实验方案保留直观的分阶段流程，同时产生额外中间结果访存。


In [ ]:
%%writefile -a Source/04.03/ascend_ops/op_kernel/rmsnorm_static_tensor.cpp

extern "C" __global__ __aicore__ void rmsnorm_basic_square(GM_ADDR x,
                                                            GM_ADDR square,
                                                            uint32_t rowCount,
                                                            uint32_t rowWidth,
                                                            uint32_t paddedRowWidth,
                                                            uint32_t numTilesPerRow,
                                                            uint32_t tileLen,
                                                            uint32_t launchBlockDim) {
    InitSocState();

    if (tileLen == 0 || tileLen > kStaticMaxTileLen || rowWidth == 0) {
        return;
    }

    const uint32_t totalLength = rowCount * paddedRowWidth;
    GlobalTensor<float> xGm;
    GlobalTensor<float> squareGm;
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(x), totalLength);
    squareGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(square), totalLength);

    uint32_t beginRow = 0;
    uint32_t endRow = 0;
    GetRowRange(rowCount, launchBlockDim, beginRow, endRow);
    if (beginRow >= endRow) {
        return;
    }

    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> tileLocal = ubAllocator.Alloc<float, kStaticMaxTileLen>();
    tileLocal.SetSize(kStaticMaxTileLen);

    for (uint32_t row = beginRow; row < endRow; ++row) {
        const uint32_t rowBase = row * paddedRowWidth;
        for (uint32_t tileId = 0; tileId < numTilesPerRow; ++tileId) {
            const uint32_t base = rowBase + tileId * tileLen;
            DataCopy(tileLocal, xGm[base], tileLen);
            PipeBarrier<PIPE_ALL>();
            Mul(tileLocal, tileLocal, tileLocal, tileLen);
            PipeBarrier<PIPE_ALL>();
            DataCopy(squareGm[base], tileLocal, tileLen);
            PipeBarrier<PIPE_ALL>();
        }
    }
}

extern "C" __global__ __aicore__ void rmsnorm_basic_write(GM_ADDR x,
                                                           GM_ADDR gamma,
                                                           GM_ADDR square,
                                                           GM_ADDR y,
                                                           uint32_t rowCount,
                                                           uint32_t rowWidth,
                                                           uint32_t paddedRowWidth,
                                                           uint32_t numTilesPerRow,
                                                           uint32_t tileLen,
                                                           float epsilon,
                                                           float invRowWidth,
                                                           uint32_t launchBlockDim) {
    InitSocState();

    if (tileLen == 0 || tileLen > kStaticMaxTileLen || rowWidth == 0) {
        return;
    }

    const uint32_t totalLength = rowCount * paddedRowWidth;
    GlobalTensor<float> xGm;
    GlobalTensor<float> gammaGm;
    GlobalTensor<float> squareGm;
    GlobalTensor<float> yGm;
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(x), totalLength);
    gammaGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(gamma), paddedRowWidth);
    squareGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(square), totalLength);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(y), totalLength);

    uint32_t beginRow = 0;
    uint32_t endRow = 0;
    GetRowRange(rowCount, launchBlockDim, beginRow, endRow);
    if (beginRow >= endRow) {
        return;
    }

    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> tileLocal = ubAllocator.Alloc<float, kStaticMaxTileLen>();
    LocalTensor<float> gammaLocal = ubAllocator.Alloc<float, kStaticMaxTileLen>();
    tileLocal.SetSize(kStaticMaxTileLen);
    gammaLocal.SetSize(kStaticMaxTileLen);

    for (uint32_t row = beginRow; row < endRow; ++row) {
        const uint32_t rowBase = row * paddedRowWidth;
        float squareSum = 0.0f;
        for (uint32_t tileId = 0; tileId < numTilesPerRow; ++tileId) {
            const uint32_t base = rowBase + tileId * tileLen;
            DataCopy(tileLocal, squareGm[base], tileLen);
            PipeBarrier<PIPE_ALL>();
            squareSum += SumLocal(tileLocal, tileLen);
        }

        const float meanSquare = squareSum * invRowWidth;
        const float invRms = InvSqrt(meanSquare + epsilon);
        for (uint32_t tileId = 0; tileId < numTilesPerRow; ++tileId) {
            const uint32_t base = rowBase + tileId * tileLen;
            const uint32_t gammaBase = tileId * tileLen;
            DataCopy(tileLocal, xGm[base], tileLen);
            DataCopy(gammaLocal, gammaGm[gammaBase], tileLen);
            PipeBarrier<PIPE_ALL>();
            ScaleAndWeightVector(tileLocal, gammaLocal, invRms, tileLen);
            PipeBarrier<PIPE_ALL>();
            DataCopy(yGm[base], tileLocal, tileLen);
            PipeBarrier<PIPE_ALL>();
        }
    }
}


### 4.3 单Kernel连续计算实验方案：融合与片上归约

`rmsnorm_fused_static_tensor`在一个Kernel中完成两遍处理。第一遍使用Vector归约工作区得到整行平方和，第二遍完成输出计算。它不缓存整行X，也不跨行预加载gamma，因此GM访问模型与双缓冲流水实验方案保持一致。


In [ ]:
%%writefile -a Source/04.03/ascend_ops/op_kernel/rmsnorm_static_tensor.cpp

extern "C" __global__ __aicore__ void rmsnorm_fused_static_tensor(GM_ADDR x,
                                                                   GM_ADDR gamma,
                                                                   GM_ADDR y,
                                                                   uint32_t rowCount,
                                                                   uint32_t rowWidth,
                                                                   uint32_t paddedRowWidth,
                                                                   uint32_t numTilesPerRow,
                                                                   uint32_t tileLen,
                                                                   float epsilon,
                                                                   float invRowWidth,
                                                                   uint32_t launchBlockDim) {
    InitSocState();

    if (tileLen == 0 || tileLen > kStaticMaxTileLen || rowWidth == 0) {
        return;
    }

    const uint32_t totalLength = rowCount * paddedRowWidth;
    GlobalTensor<float> xGm;
    GlobalTensor<float> gammaGm;
    GlobalTensor<float> yGm;
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(x), totalLength);
    gammaGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(gamma), paddedRowWidth);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(y), totalLength);

    uint32_t beginRow = 0;
    uint32_t endRow = 0;
    GetRowRange(rowCount, launchBlockDim, beginRow, endRow);
    if (beginRow >= endRow) {
        return;
    }

    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> tileLocal = ubAllocator.Alloc<float>(tileLen);
    LocalTensor<float> gammaLocal = ubAllocator.Alloc<float>(tileLen);
    LocalTensor<float> yLocal = ubAllocator.Alloc<float>(tileLen);
    LocalTensor<float> reduceLocal = ubAllocator.Alloc<float, kReduceWorkspaceLen>();
    tileLocal.SetSize(tileLen);
    gammaLocal.SetSize(tileLen);
    yLocal.SetSize(tileLen);
    reduceLocal.SetSize(kReduceWorkspaceLen);

    for (uint32_t row = beginRow; row < endRow; ++row) {
        const uint32_t rowBase = row * paddedRowWidth;
        float squareSum = 0.0f;
        for (uint32_t tileId = 0; tileId < numTilesPerRow; ++tileId) {
            const uint32_t base = rowBase + tileId * tileLen;
            DataCopy(tileLocal, xGm[base], tileLen);
            PipeBarrier<PIPE_ALL>();
            squareSum += SquareAndReduceVector(tileLocal, reduceLocal, tileLen);
        }

        const float meanSquare = squareSum * invRowWidth;
        const float invRms = InvSqrt(meanSquare + epsilon);
        for (uint32_t tileId = 0; tileId < numTilesPerRow; ++tileId) {
            const uint32_t base = rowBase + tileId * tileLen;
            const uint32_t gammaBase = tileId * tileLen;
            DataCopy(tileLocal, xGm[base], tileLen);
            DataCopy(gammaLocal, gammaGm[gammaBase], tileLen);
            PipeBarrier<PIPE_ALL>();
            ScaleAndWeightVectorToOut(yLocal, tileLocal, gammaLocal, invRms, tileLen);
            PipeBarrier<PIPE_ALL>();
            DataCopy(yGm[base], yLocal, tileLen);
            PipeBarrier<PIPE_ALL>();
        }
    }
}


### 4.4 双缓冲流水实验方案：辅助阶段

双缓冲流水实验方案把单个tile的操作拆成四类辅助函数：

- `PipelineEnqueueX`：申请输入缓冲并搬入X；
- `PipelineEnqueueXGamma`：并行准备第二遍所需的X和gamma；
- `PipelineDequeueAndReduce/Scale`：从队列取出数据并执行Vector计算；
- `PipelineDequeueAndCopyOut`：取出输出并写回GM。

`AllocTensor→DataCopy→EnQue`建立生产阶段，`DeQue→Compute/CopyOut→FreeTensor`建立消费阶段。


In [ ]:
%%writefile -a Source/04.03/ascend_ops/op_kernel/rmsnorm_static_tensor.cpp

__aicore__ inline void PipelineEnqueueX(PipelineInQueue& xQueue,
                                        GlobalTensor<float> xGm,
                                        uint32_t base,
                                        uint32_t len) {
    LocalTensor<float> xLocal = xQueue.AllocTensor<float>();
    DataCopy(xLocal, xGm[base], len);
    xQueue.EnQue(xLocal);
}

__aicore__ inline void PipelineEnqueueXGamma(PipelineInQueue& xQueue,
                                             PipelineInQueue& gammaQueue,
                                             GlobalTensor<float> xGm,
                                             GlobalTensor<float> gammaGm,
                                             uint32_t xBase,
                                             uint32_t gammaBase,
                                             uint32_t len) {
    LocalTensor<float> xLocal = xQueue.AllocTensor<float>();
    LocalTensor<float> gammaLocal = gammaQueue.AllocTensor<float>();
    DataCopy(xLocal, xGm[xBase], len);
    DataCopy(gammaLocal, gammaGm[gammaBase], len);
    xQueue.EnQue(xLocal);
    gammaQueue.EnQue(gammaLocal);
}

__aicore__ inline float PipelineDequeueAndReduce(PipelineInQueue& xQueue,
                                                 LocalTensor<float> reduceLocal,
                                                 uint32_t len) {
    LocalTensor<float> xLocal = xQueue.DeQue<float>();
    const float tileSum = SquareAndReduceVector(xLocal, reduceLocal, len);
    xQueue.FreeTensor(xLocal);
    return tileSum;
}

__aicore__ inline void PipelineDequeueAndScale(PipelineInQueue& xQueue,
                                               PipelineInQueue& gammaQueue,
                                               PipelineOutQueue& yQueue,
                                               float invRms,
                                               uint32_t len) {
    LocalTensor<float> xLocal = xQueue.DeQue<float>();
    LocalTensor<float> gammaLocal = gammaQueue.DeQue<float>();
    LocalTensor<float> yLocal = yQueue.AllocTensor<float>();
    ScaleAndWeightVectorToOut(yLocal, xLocal, gammaLocal, invRms, len);
    yQueue.EnQue(yLocal);
    xQueue.FreeTensor(xLocal);
    gammaQueue.FreeTensor(gammaLocal);
}

__aicore__ inline void PipelineDequeueAndCopyOut(PipelineOutQueue& yQueue,
                                                 GlobalTensor<float> yGm,
                                                 uint32_t base,
                                                 uint32_t len) {
    LocalTensor<float> yLocal = yQueue.DeQue<float>();
    DataCopy(yGm[base], yLocal, len);
    yQueue.FreeTensor(yLocal);
}


### 4.5 双缓冲流水实验方案：Kernel实现

双缓冲流水实验方案为X、gamma和Y分别建立深度为2的TQue，并为Vector归约建立独立TBuf。每一遍开始时先预取最多两个tile；循环中消费当前tile后再补入后续tile，从而维持队列中的流水数据。

第一遍完全结束后才计算`invRms`并进入第二遍，这一顺序严格满足RMSNorm的整行归约依赖。


In [ ]:
%%writefile -a Source/04.03/ascend_ops/op_kernel/rmsnorm_static_tensor.cpp

extern "C" __global__ __aicore__ void rmsnorm_pipeline_static_tensor(GM_ADDR x,
                                                                      GM_ADDR gamma,
                                                                      GM_ADDR y,
                                                                      uint32_t rowCount,
                                                                      uint32_t rowWidth,
                                                                      uint32_t paddedRowWidth,
                                                                      uint32_t numTilesPerRow,
                                                                      uint32_t tileLen,
                                                                      float epsilon,
                                                                      float invRowWidth,
                                                                      uint32_t launchBlockDim) {
    InitSocState();

    if (tileLen == 0 || tileLen > kPipelineMaxTileLen || rowWidth == 0) {
        return;
    }

    const uint32_t totalLength = rowCount * paddedRowWidth;
    GlobalTensor<float> xGm;
    GlobalTensor<float> gammaGm;
    GlobalTensor<float> yGm;
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(x), totalLength);
    gammaGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(gamma), paddedRowWidth);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(y), totalLength);

    uint32_t beginRow = 0;
    uint32_t endRow = 0;
    GetRowRange(rowCount, launchBlockDim, beginRow, endRow);
    if (beginRow >= endRow) {
        return;
    }

    // Fused and pipeline share the same tile math and GM traffic. Only this
    // path owns two buffers per stage and schedules adjacent tiles through
    // VECIN/VECOUT queues so MTE2, vector, and MTE3 can overlap.
    TPipe pipe;
    PipelineInQueue xQueue;
    PipelineInQueue gammaQueue;
    PipelineOutQueue yQueue;
    TBuf<TPosition::VECCALC> reduceBuffer;
    pipe.InitBuffer(xQueue, kPipelineBufferNum, tileLen * sizeof(float));
    pipe.InitBuffer(gammaQueue, kPipelineBufferNum, tileLen * sizeof(float));
    pipe.InitBuffer(yQueue, kPipelineBufferNum, tileLen * sizeof(float));
    pipe.InitBuffer(reduceBuffer, kReduceWorkspaceLen * sizeof(float));
    LocalTensor<float> reduceLocal = reduceBuffer.Get<float>();
    reduceLocal.SetSize(kReduceWorkspaceLen);
    const uint32_t prefetchCount =
        numTilesPerRow < static_cast<uint32_t>(kPipelineBufferNum)
            ? numTilesPerRow
            : static_cast<uint32_t>(kPipelineBufferNum);

    for (uint32_t row = beginRow; row < endRow; ++row) {
        const uint32_t rowBase = row * paddedRowWidth;

        float squareSum = 0.0f;
        for (uint32_t tileId = 0; tileId < prefetchCount; ++tileId) {
            PipelineEnqueueX(xQueue, xGm, rowBase + tileId * tileLen, tileLen);
        }
        for (uint32_t tileId = 0; tileId < numTilesPerRow; ++tileId) {
            squareSum += PipelineDequeueAndReduce(xQueue, reduceLocal, tileLen);
            const uint32_t nextTile = tileId + prefetchCount;
            if (nextTile < numTilesPerRow) {
                PipelineEnqueueX(xQueue, xGm, rowBase + nextTile * tileLen, tileLen);
            }
        }

        const float meanSquare = squareSum * invRowWidth;
        const float invRms = InvSqrt(meanSquare + epsilon);

        for (uint32_t tileId = 0; tileId < prefetchCount; ++tileId) {
            const uint32_t base = rowBase + tileId * tileLen;
            PipelineEnqueueXGamma(xQueue, gammaQueue, xGm, gammaGm,
                                  base, tileId * tileLen, tileLen);
        }
        for (uint32_t tileId = 0; tileId < numTilesPerRow; ++tileId) {
            const uint32_t base = rowBase + tileId * tileLen;
            PipelineDequeueAndScale(xQueue, gammaQueue, yQueue, invRms, tileLen);

            const uint32_t nextTile = tileId + prefetchCount;
            if (nextTile < numTilesPerRow) {
                PipelineEnqueueXGamma(xQueue, gammaQueue, xGm, gammaGm,
                                      rowBase + nextTile * tileLen,
                                      nextTile * tileLen, tileLen);
            }
            PipelineDequeueAndCopyOut(yQueue, yGm, base, tileLen);
        }
    }
}


---
## 5. 结果验证与性能分析

### 5.1 Host参考结果与误差校验

Host侧工具负责生成固定输入和缩放权重，按照RMSNorm数学公式计算参考结果，并比较Device输出。它只承担输入准备和正确性校验，不参与Device侧三种实验方案的性能计时。

误差判定同时使用绝对误差和相对误差阈值：

$$
|y-ref|>atol+rtol\cdot|ref|
$$

满足该条件的元素计入`errors`。


In [ ]:
%%writefile Source/04.03/include/rmsnorm_host_reference.h

#pragma once

#include "rmsnorm_common.h"

#include <algorithm>
#include <cmath>
#include <iostream>
#include <random>
#include <stdexcept>
#include <vector>

namespace rmsnorm {

inline std::vector<float> make_input(uint32_t m, uint32_t n, uint32_t seed) {
    check_config(m, n, 8, sizeof(float));
    std::mt19937 rng(seed);
    std::uniform_real_distribution<float> dist(-2.0f, 2.0f);
    std::vector<float> x(valid_element_count(m, n));
    for (float& value : x) {
        value = dist(rng);
    }
    if (x.size() >= 8) {
        x[0] = -1.50f;
        x[1] = -0.25f;
        x[2] = 0.0f;
        x[3] = 0.125f;
        x[4] = 1.0f;
        x[5] = -2.0f;
        x[6] = 3.5f;
        x[7] = -0.001f;
    }
    return x;
}

inline std::vector<float> make_weight(uint32_t n, uint32_t seed) {
    if (n == 0) {
        throw std::invalid_argument("n must be > 0");
    }
    std::mt19937 rng(seed + 17u);
    std::uniform_real_distribution<float> dist(0.75f, 1.25f);
    std::vector<float> gamma(n);
    for (float& value : gamma) {
        value = dist(rng);
    }
    if (n >= 8) {
        gamma[0] = 1.0f;
        gamma[1] = 0.5f;
        gamma[2] = 1.5f;
        gamma[3] = 0.75f;
        gamma[4] = 1.25f;
        gamma[5] = 1.0f;
        gamma[6] = 0.875f;
        gamma[7] = 1.125f;
    }
    return gamma;
}

inline std::vector<float> rmsnorm_reference(const std::vector<float>& x,
                                            const std::vector<float>& gamma,
                                            uint32_t m,
                                            uint32_t n,
                                            float epsilon) {
    if (x.size() != valid_element_count(m, n) || gamma.size() != n) {
        throw std::invalid_argument("rmsnorm_reference input size mismatch");
    }
    if (!(epsilon > 0.0f)) {
        throw std::invalid_argument("epsilon must be positive");
    }

    std::vector<float> y(x.size(), 0.0f);
    for (uint32_t row = 0; row < m; ++row) {
        const uint64_t rowBase = static_cast<uint64_t>(row) * n;
        double squareSum = 0.0;
        for (uint32_t col = 0; col < n; ++col) {
            const double value = static_cast<double>(x[rowBase + col]);
            squareSum += value * value;
        }
        const double meanSquare = squareSum / static_cast<double>(n);
        const float invRms = static_cast<float>(
            1.0 / std::sqrt(meanSquare + static_cast<double>(epsilon)));
        for (uint32_t col = 0; col < n; ++col) {
            y[rowBase + col] = x[rowBase + col] * invRms * gamma[col];
        }
    }
    return y;
}

inline Metrics compare_vectors(const std::vector<float>& got,
                               const std::vector<float>& ref,
                               double atol = 1.0e-4,
                               double rtol = 1.0e-4) {
    if (got.size() != ref.size()) {
        throw std::invalid_argument("compare_vectors size mismatch");
    }
    Metrics metrics;
    for (size_t i = 0; i < got.size(); ++i) {
        const double g = static_cast<double>(got[i]);
        const double r = static_cast<double>(ref[i]);
        const double absError = std::abs(g - r);
        const double relError = absError / std::max(1.0, std::abs(r));
        metrics.max_abs_error = std::max(metrics.max_abs_error, absError);
        metrics.max_rel_error = std::max(metrics.max_rel_error, relError);
        if (absError > atol + rtol * std::abs(r)) {
            ++metrics.mismatch_count;
        }
    }
    return metrics;
}

inline void dump_sample(const std::vector<float>& x,
                        const std::vector<float>& gamma,
                        const std::vector<float>& y,
                        const std::vector<float>& ref,
                        uint32_t m,
                        uint32_t n,
                        size_t count = 8) {
    const size_t validCount = static_cast<size_t>(valid_element_count(m, n));
    const size_t outCount = std::min({count, validCount, y.size(), ref.size()});
    std::cout << "sample(first row, first " << outCount << "):\n";
    for (size_t i = 0; i < outCount; ++i) {
        std::cout << "  col=" << i
                  << " x=" << x[i]
                  << " gamma=" << gamma[i]
                  << " y=" << y[i]
                  << " ref=" << ref[i]
                  << "\n";
    }
}

}  // namespace rmsnorm


### 5.2 参数解析与ACL资源管理

Host程序首先解析设备编号、M、N、tileLen、blockDim、epsilon、实验方案标识、预热次数和重复次数。`DeviceBuffer`与`StreamGuard`使用析构函数统一释放Device内存和stream，异常路径也能完成资源清理。


In [ ]:
%%writefile Source/04.03/ascend_ops/host_launch/rmsnorm_npu_main.cpp

#include <acl/acl.h>
#include <aclrtlaunch_rmsnorm_basic_square.h>
#include <aclrtlaunch_rmsnorm_basic_write.h>
#include <aclrtlaunch_rmsnorm_fused_static_tensor.h>
#include <aclrtlaunch_rmsnorm_pipeline_static_tensor.h>

#include "rmsnorm_host_reference.h"

#include <cstdint>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <limits>
#include <stdexcept>
#include <string>
#include <vector>

#define ACL_CHECK(expr)                                                                     \
    do {                                                                                    \
        aclError _ret = (expr);                                                            \
        if (_ret != ACL_SUCCESS) {                                                         \
            throw std::runtime_error(std::string("ACL error: ") + #expr +                 \
                                     ", code=" + std::to_string(static_cast<int>(_ret))); \
        }                                                                                   \
    } while (0)

struct NpuConfig : rmsnorm::Config {
    int32_t device = 0;
    uint32_t warmup = 2;
    uint32_t repeat = 10;
};

struct DeviceBuffer {
    void* ptr = nullptr;
    ~DeviceBuffer() {
        if (ptr != nullptr) {
            (void)aclrtFree(ptr);
        }
    }
    DeviceBuffer() = default;
    DeviceBuffer(const DeviceBuffer&) = delete;
    DeviceBuffer& operator=(const DeviceBuffer&) = delete;
};

struct StreamGuard {
    aclrtStream stream = nullptr;
    ~StreamGuard() {
        if (stream != nullptr) {
            (void)aclrtDestroyStream(stream);
        }
    }
    StreamGuard() = default;
    StreamGuard(const StreamGuard&) = delete;
    StreamGuard& operator=(const StreamGuard&) = delete;
};

static void usage(const char* argv0) {
    std::cout << "Usage: " << argv0 << " [options]\n"
              << "Options:\n"
              << "  --device <id>       device id, default: 0\n"
              << "  --m <num>           row count, default: 256\n"
              << "  --n <num>           row width, default: 1024\n"
              << "  --tile-len <num>    static Tensor tile length, default: 256\n"
              << "  --block-dim <num>   AI Core launch blockDim, default: 8\n"
              << "  --epsilon <value>   RMSNorm epsilon, default: 1e-5\n"
              << "  --version <name>    basic/fused/pipeline, default: fused\n"
              << "  --warmup <num>      warmup count, default: 2\n"
              << "  --repeat <num>      repeat count, default: 10\n"
              << "  --seed <num>        random seed, default: 1234\n"
              << "  --sweep             test tileLen = 64/128/256/512/1024/2048\n"
              << "  --print-output      print first 8 output values\n"
              << "  -h, --help          show help\n";
}

static NpuConfig parse_args(int argc, char** argv) {
    NpuConfig cfg;
    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto need_value = [&](const std::string& name) -> const char* {
            if (i + 1 >= argc) {
                throw std::invalid_argument("missing value after " + name);
            }
            return argv[++i];
        };
        if (arg == "--device") {
            cfg.device = std::stoi(need_value(arg));
        } else if (arg == "--m") {
            cfg.m = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--n") {
            cfg.n = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--tile-len") {
            cfg.tile_len = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--block-dim") {
            cfg.block_dim = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--epsilon") {
            cfg.epsilon = std::stof(need_value(arg));
        } else if (arg == "--version") {
            cfg.version = rmsnorm::parse_version(need_value(arg));
        } else if (arg == "--warmup") {
            cfg.warmup = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--repeat") {
            cfg.repeat = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--seed") {
            cfg.seed = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--sweep") {
            cfg.sweep = true;
        } else if (arg == "--print-output") {
            cfg.print_output = true;
        } else if (arg == "-h" || arg == "--help") {
            usage(argv[0]);
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    if (cfg.block_dim == 0) {
        throw std::invalid_argument("--block-dim must be positive");
    }
    if (cfg.repeat == 0) {
        throw std::invalid_argument("--repeat must be positive");
    }
    if (!(cfg.epsilon > 0.0f)) {
        throw std::invalid_argument("--epsilon must be positive");
    }
    return cfg;
}


### 5.3 补齐数据的准备与恢复

Host按照`paddedN`建立补齐后的X、gamma和Y。有效数据复制到每行前N个位置，补齐部分置0；Device结果回传后只提取每行前N个有效元素进行校验。


In [ ]:
%%writefile -a Source/04.03/ascend_ops/host_launch/rmsnorm_npu_main.cpp

static void copy_to_padded(const std::vector<float>& x,
                           uint32_t m,
                           uint32_t n,
                           uint32_t paddedN,
                           std::vector<float>& paddedX) {
    paddedX.assign(rmsnorm::padded_element_count(m, paddedN), 0.0f);
    for (uint32_t row = 0; row < m; ++row) {
        const uint64_t srcBase = static_cast<uint64_t>(row) * n;
        const uint64_t dstBase = static_cast<uint64_t>(row) * paddedN;
        std::copy_n(x.begin() + static_cast<std::ptrdiff_t>(srcBase),
                    n,
                    paddedX.begin() + static_cast<std::ptrdiff_t>(dstBase));
    }
}

static void copy_from_padded(const std::vector<float>& paddedY,
                             uint32_t m,
                             uint32_t n,
                             uint32_t paddedN,
                             std::vector<float>& y) {
    y.assign(rmsnorm::valid_element_count(m, n), 0.0f);
    for (uint32_t row = 0; row < m; ++row) {
        const uint64_t srcBase = static_cast<uint64_t>(row) * paddedN;
        const uint64_t dstBase = static_cast<uint64_t>(row) * n;
        std::copy_n(paddedY.begin() + static_cast<std::ptrdiff_t>(srcBase),
                    n,
                    y.begin() + static_cast<std::ptrdiff_t>(dstBase));
    }
}


### 5.4 Device内存申请与实验初始化

程序根据所选实验方案申请X、gamma、Y和可选的平方中间缓冲。基线实验方案（`basic`）需要在GM中额外申请一块中间缓冲区，单Kernel连续计算实验方案（`fused`）和双缓冲流水实验方案（`pipeline`）不申请该缓冲。固定输入只在正式测量前上传一次，避免把Host到Device输入传输计入三种Kernel组织的对比。


In [ ]:
%%writefile -a Source/04.03/ascend_ops/host_launch/rmsnorm_npu_main.cpp

static void run_rmsnorm_static_tensor(const NpuConfig& cfg, uint32_t tileLen) {
    rmsnorm::check_config(cfg.m, cfg.n, tileLen, sizeof(float));
    constexpr uint32_t kStaticMaxTileLen = 8192;
    constexpr uint32_t kPipelineMaxTileLen = 4096;
    if (tileLen > kStaticMaxTileLen) {
        throw std::invalid_argument("--tile-len exceeds static Tensor capacity 8192");
    }
    if (cfg.version == rmsnorm::Version::Pipeline && tileLen > kPipelineMaxTileLen) {
        throw std::invalid_argument(
            "--tile-len exceeds double-buffer pipeline capacity 4096");
    }

    const uint32_t paddedN = static_cast<uint32_t>(rmsnorm::round_up(cfg.n, tileLen));
    const uint32_t numTilesPerRow = rmsnorm::ceil_div_u32(cfg.n, tileLen);
    const uint64_t paddedElements = rmsnorm::padded_element_count(cfg.m, paddedN);
    if (paddedElements > static_cast<uint64_t>(std::numeric_limits<uint32_t>::max())) {
        throw std::invalid_argument("padded total length exceeds uint32_t kernel argument range");
    }

    const size_t xBytes = rmsnorm::checked_float_bytes(paddedElements);
    const size_t gammaBytes = rmsnorm::checked_float_bytes(paddedN);
    const float invRowWidth = 1.0f / static_cast<float>(cfg.n);

    std::vector<float> xValid = rmsnorm::make_input(cfg.m, cfg.n, cfg.seed);
    std::vector<float> gammaValid = rmsnorm::make_weight(cfg.n, cfg.seed);
    std::vector<float> ref = rmsnorm::rmsnorm_reference(xValid, gammaValid, cfg.m, cfg.n, cfg.epsilon);
    std::vector<float> xPadded;
    copy_to_padded(xValid, cfg.m, cfg.n, paddedN, xPadded);
    std::vector<float> gammaPadded(paddedN, 0.0f);
    std::copy(gammaValid.begin(), gammaValid.end(), gammaPadded.begin());
    std::vector<float> yPadded(paddedElements, 0.0f);

    DeviceBuffer xDevice;
    DeviceBuffer gammaDevice;
    DeviceBuffer yDevice;
    DeviceBuffer squareDevice;
    StreamGuard stream;

    ACL_CHECK(aclrtCreateStream(&stream.stream));
    ACL_CHECK(aclrtMalloc(&xDevice.ptr, xBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&gammaDevice.ptr, gammaBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&yDevice.ptr, xBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    if (cfg.version == rmsnorm::Version::Basic) {
        ACL_CHECK(aclrtMalloc(&squareDevice.ptr, xBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    }
    ACL_CHECK(aclrtMemcpy(xDevice.ptr, xBytes, xPadded.data(), xBytes, ACL_MEMCPY_HOST_TO_DEVICE));
    ACL_CHECK(aclrtMemcpy(gammaDevice.ptr, gammaBytes, gammaPadded.data(), gammaBytes, ACL_MEMCPY_HOST_TO_DEVICE));


### 5.5 Kernel启动、预热与重复测量

`run_once`根据所选实验方案启动对应Kernel。基线实验方案连续启动平方Kernel和输出Kernel；另外两种实验方案各启动一个Kernel。每次启动后同步stream，再累计`kernel_us`和`total_us`。先执行warmup次预热，再重复repeat次并计算平均值。


In [ ]:
%%writefile -a Source/04.03/ascend_ops/host_launch/rmsnorm_npu_main.cpp

    auto run_once = [&](bool timed, rmsnorm::TimingUs* timing) {
        rmsnorm::Timer totalTimer;
        // Each RMSNorm kernel writes all padded output elements. The basic square kernel
        // also overwrites the full square buffer. Therefore timed memset is unnecessary
        // and would make total_us include extra memory initialization that is not part
        // of the RMSNorm algorithm itself.
        rmsnorm::Timer kernelTimer;
        if (cfg.version == rmsnorm::Version::Basic) {
            ACLRT_LAUNCH_KERNEL(rmsnorm_basic_square)(cfg.block_dim, stream.stream,
                                                      xDevice.ptr, squareDevice.ptr,
                                                      cfg.m, cfg.n, paddedN, numTilesPerRow,
                                                      tileLen, cfg.block_dim);
            ACLRT_LAUNCH_KERNEL(rmsnorm_basic_write)(cfg.block_dim, stream.stream,
                                                     xDevice.ptr, gammaDevice.ptr, squareDevice.ptr, yDevice.ptr,
                                                     cfg.m, cfg.n, paddedN, numTilesPerRow,
                                                     tileLen, cfg.epsilon, invRowWidth, cfg.block_dim);
        } else if (cfg.version == rmsnorm::Version::Fused) {
            ACLRT_LAUNCH_KERNEL(rmsnorm_fused_static_tensor)(cfg.block_dim, stream.stream,
                                                             xDevice.ptr, gammaDevice.ptr, yDevice.ptr,
                                                             cfg.m, cfg.n, paddedN, numTilesPerRow,
                                                             tileLen, cfg.epsilon, invRowWidth, cfg.block_dim);
        } else {
            ACLRT_LAUNCH_KERNEL(rmsnorm_pipeline_static_tensor)(cfg.block_dim, stream.stream,
                                                                xDevice.ptr, gammaDevice.ptr, yDevice.ptr,
                                                                cfg.m, cfg.n, paddedN, numTilesPerRow,
                                                                tileLen, cfg.epsilon, invRowWidth, cfg.block_dim);
        }
        ACL_CHECK(aclrtSynchronizeStream(stream.stream));
        if (timed) {
            timing->kernel_us += kernelTimer.elapsed_us();
            timing->total_us += totalTimer.elapsed_us();
        }
    };

    rmsnorm::TimingUs timing;
    for (uint32_t i = 0; i < cfg.warmup; ++i) {
        run_once(false, &timing);
    }
    for (uint32_t i = 0; i < cfg.repeat; ++i) {
        run_once(true, &timing);
    }
    timing.kernel_us /= cfg.repeat;
    timing.total_us /= cfg.repeat;

    ACL_CHECK(aclrtMemcpy(yPadded.data(), xBytes, yDevice.ptr, xBytes, ACL_MEMCPY_DEVICE_TO_HOST));
    std::vector<float> yValid;
    copy_from_padded(yPadded, cfg.m, cfg.n, paddedN, yValid);
    const rmsnorm::Metrics metrics = rmsnorm::compare_vectors(yValid, ref, 1.0e-4, 1.0e-4);

    rmsnorm::print_result_row(cfg.m, cfg.n, paddedN, tileLen, numTilesPerRow,
                              cfg.block_dim, cfg.version, timing, metrics);
    std::cout << "  version=rmsnorm_static_tensor"
              << ", mode=" << rmsnorm::version_name(cfg.version)
              << ", launchBlockDim=" << cfg.block_dim
              << ", tileLen=" << tileLen
              << ", epsilon=" << cfg.epsilon
              << ", warmup=" << cfg.warmup
              << ", repeat=" << cfg.repeat
              << ", estimatedMovedBytes=" << static_cast<uint64_t>(rmsnorm::estimated_moved_bytes(paddedElements, cfg.version)) << "\n";

    if (cfg.print_output) {
        rmsnorm::dump_sample(xValid, gammaValid, yValid, ref, cfg.m, cfg.n, 8);
    }
}


### 5.6 结果回传、指标输出与主流程

正式测量结束后，Host回传Y并与参考结果比较。输出包含尺寸、补齐行宽、tile数量、核心数、实验方案标识、平均耗时、估算有效带宽、误差、错误数量和状态。主流程负责ACL初始化、Device选择、普通运行或tileLen扫描以及最终清理。


In [ ]:
%%writefile -a Source/04.03/ascend_ops/host_launch/rmsnorm_npu_main.cpp

int main(int argc, char** argv) {
    NpuConfig cfg;
    bool aclInitialized = false;
    bool deviceSet = false;

    auto cleanup_acl = [&]() noexcept {
        if (deviceSet) {
            (void)aclrtResetDevice(cfg.device);
            deviceSet = false;
        }
        if (aclInitialized) {
            (void)aclFinalize();
            aclInitialized = false;
        }
    };

    try {
        cfg = parse_args(argc, argv);
        ACL_CHECK(aclInit(nullptr));
        aclInitialized = true;
        ACL_CHECK(aclrtSetDevice(cfg.device));
        deviceSet = true;

        rmsnorm::print_header();
        if (cfg.sweep) {
            for (uint32_t tileLen : {64u, 128u, 256u, 512u, 1024u, 2048u}) {
                run_rmsnorm_static_tensor(cfg, tileLen);
            }
        } else {
            run_rmsnorm_static_tensor(cfg, cfg.tile_len);
        }

        cleanup_acl();
        return 0;
    } catch (const std::exception& e) {
        cleanup_acl();
        std::cerr << "error: " << e.what() << "\n";
        usage(argv[0]);
        return 1;
    }
}


### 5.7 Kernel启动头文件示例

Ascend C构建过程会根据四个核函数生成对应的`aclrtlaunch_*.h`。下面的小文件集中展示Host侧需要包含的自动生成头文件，可用于核对Kernel名称。


In [ ]:
%%writefile Source/04.03/ascend_ops/host_launch/rmsnorm_launch_example.cpp

#include <acl/acl.h>
#include <aclrtlaunch_rmsnorm_basic_square.h>
#include <aclrtlaunch_rmsnorm_basic_write.h>
#include <aclrtlaunch_rmsnorm_fused_static_tensor.h>
#include <aclrtlaunch_rmsnorm_pipeline_static_tensor.h>


### 5.8 工程构建

CMake先通过`ascendc_library`编译Device核函数，再编译NPU Host程序并链接`ascendcl`。Host参考工具采用头文件内联实现，工程仅生成NPU实验可执行程序。


In [ ]:
%%writefile Source/04.03/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
project(ascendc_static_tensor_rmsnorm LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

option(BUILD_ASCEND "Build Ascend C NPU demo" OFF)

if(BUILD_ASCEND)
  set(RUN_MODE "npu" CACHE STRING "Ascend C run mode")
  set(SOC_VERSION "ascend910b1" CACHE STRING "Ascend SOC version")
  set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}" CACHE PATH "CANN installation path")
  if(NOT ASCEND_CANN_PATH)
    set(ASCEND_CANN_PATH "/usr/local/Ascend/ascend-toolkit/latest" CACHE PATH "CANN installation path" FORCE)
  endif()
  set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH "CANN package path" FORCE)
  set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "Ascend C install output" FORCE)

  if(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  else()
    message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}.")
  endif()

  message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
  message(STATUS "SOC_VERSION=${SOC_VERSION}")
  include("${ASCENDC_CMAKE_FILE}")

  ascendc_library(rmsnorm_kernels STATIC
      ascend_ops/op_kernel/rmsnorm_static_tensor.cpp
  )
  ascendc_compile_definitions(rmsnorm_kernels PRIVATE
      -DASCENDC_DUMP=0
  )

  add_executable(rmsnorm_ascend_demo
      ascend_ops/host_launch/rmsnorm_npu_main.cpp
  )
  target_include_directories(rmsnorm_ascend_demo PRIVATE
      include
      ${ASCEND_CANN_PACKAGE_PATH}/include
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
      ${CMAKE_INSTALL_PREFIX}/include/rmsnorm_kernels
      ${CMAKE_BINARY_DIR}/out/include/rmsnorm_kernels
  )
  target_link_directories(rmsnorm_ascend_demo PRIVATE
      ${ASCEND_CANN_PACKAGE_PATH}/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
  )
  target_link_libraries(rmsnorm_ascend_demo PRIVATE
      rmsnorm_kernels
      ascendcl
  )
  add_dependencies(rmsnorm_ascend_demo rmsnorm_kernels)
endif()

install(DIRECTORY ascend_ops DESTINATION share/ascendc_static_tensor_rmsnorm)


### 5.9 构建、运行与Profiling脚本

运行脚本负责定位CANN、配置CMake、编译工程并传递实验参数。工程的`BUILD_ASCEND`默认值为`OFF`，执行`run_ascend.sh`时由脚本显式传入`-DBUILD_ASCEND=ON`。为避免CMake增量编译复用上次残留的目标文件，下面的实验命令统一加入`-c`，每次运行前清理`build_ascend`后重新配置和编译。

Profiling脚本接收`basic`、`fused`或`pipeline`作为第一个参数，分别对应基线实验方案、单Kernel连续计算实验方案和双缓冲流水实验方案，并使用与正式实验一致的M、N、tileLen、blockDim、epsilon、warmup和repeat。


In [ ]:
%%writefile Source/04.03/scripts/run_ascend.sh

#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR=$(cd "$(dirname "$0")/.." && pwd)
BUILD_DIR="${SCRIPT_DIR}/build_ascend"
ASCEND_INSTALL_PATH_DEFAULT="/usr/local/Ascend/ascend-toolkit/latest"
ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH:-}"
SOC_VERSION="${SOC_VERSION:-ascend910b1}"

resolve_ascend_install_path() {
  if [[ -n "${ASCEND_INSTALL_PATH}" && -d "${ASCEND_INSTALL_PATH}" ]]; then
    return 0
  fi

  local candidates=()
  [[ -n "${ASCEND_TOOLKIT_HOME:-}" ]] && candidates+=("${ASCEND_TOOLKIT_HOME}")
  [[ -n "${ASCEND_HOME_PATH:-}" ]] && candidates+=("${ASCEND_HOME_PATH}")
  [[ -n "${ASCEND_CANN_PACKAGE_PATH:-}" ]] && candidates+=("${ASCEND_CANN_PACKAGE_PATH}")

  candidates+=(
    "${ASCEND_INSTALL_PATH_DEFAULT}"
    "/usr/local/Ascend/ascend-toolkit"
    "/opt/Ascend/ascend-toolkit/latest"
    "/opt/Ascend/ascend-toolkit"
    "${HOME}/Ascend/ascend-toolkit/latest"
    "${HOME}/Ascend/ascend-toolkit"
    "/workspace/Ascend/ascend-toolkit/latest"
    "/workspace/Ascend/ascend-toolkit"
  )

  local path
  for path in "${candidates[@]}"; do
    if [[ -n "${path}" && -d "${path}" ]]; then
      ASCEND_INSTALL_PATH="${path}"
      return 0
    fi
  done

  local found_set_env=""
  found_set_env=$(find /usr/local/Ascend /opt/Ascend "${HOME}" /workspace -path "*/ascend-toolkit*/set_env.sh" -print -quit 2>/dev/null || true)
  if [[ -n "${found_set_env}" ]]; then
    ASCEND_INSTALL_PATH="$(dirname "${found_set_env}")"
    return 0
  fi

  return 1
}

DEVICE_ID=0
RUN_MODE="npu"
BUILD_TYPE="Release"
CLEAN=0
WARMUP=2
REPEAT=10
M=256
N=1024
TILE_LEN=256
BLOCK_DIM=8
EPSILON="1e-5"
VERSION="fused"
SEED=1234
SWEEP=0
EXTRA_ARGS=()

usage() {
  cat <<USAGE
Usage: bash scripts/run_ascend.sh [options] [-- extra_args_for_binary]

Options:
  -a <path>   ASCEND_INSTALL_PATH. If omitted, the script auto-detects ascend-toolkit.
  -v <soc>    SOC_VERSION, default: ascend910b1. Example: ascend910b2, ascend910b3, ascend310p3
  -d <id>     device id, default: 0
  -M <num>    row count M, default: 256
  -N <num>    row width N, default: 1024
  -l <num>    static Tensor tile length, default: 256
  -b <num>    AI Core launch blockDim, default: 8
  -p <value>  epsilon, default: 1e-5
  -V <name>   version: basic, fused, or pipeline. default: fused
  -w <num>    warmup count, default: 2
  -r <num>    repeat count, default: 10
  -e <num>    random seed, default: 1234
  -m <mode>   CMake run mode, default: npu. Keep npu for Ascend execution.
  -t <type>   CMake build type, default: Release
  -s          sweep tileLen = 64/128/256/512/1024/2048
  -c          clean build directory before building
  -h          show help

Examples:
  bash scripts/run_ascend.sh -c
  bash scripts/run_ascend.sh -c -a /usr/local/Ascend/ascend-toolkit/latest -v ascend910b1 -d 0
  bash scripts/run_ascend.sh -c -M 256 -N 1024 -l 256 -b 8 -V fused -w 2 -r 10
  bash scripts/run_ascend.sh -c -V basic
  bash scripts/run_ascend.sh -c -V pipeline -s
  bash scripts/run_ascend.sh -c -- --print-output
USAGE
}

while getopts ":a:v:d:M:N:l:b:p:V:w:r:e:m:t:sch" opt; do
  case ${opt} in
    a) ASCEND_INSTALL_PATH="${OPTARG}" ;;
    v) SOC_VERSION="${OPTARG}" ;;
    d) DEVICE_ID="${OPTARG}" ;;
    M) M="${OPTARG}" ;;
    N) N="${OPTARG}" ;;
    l) TILE_LEN="${OPTARG}" ;;
    b) BLOCK_DIM="${OPTARG}" ;;
    p) EPSILON="${OPTARG}" ;;
    V) VERSION="${OPTARG}" ;;
    w) WARMUP="${OPTARG}" ;;
    r) REPEAT="${OPTARG}" ;;
    e) SEED="${OPTARG}" ;;
    m) RUN_MODE="${OPTARG}" ;;
    t) BUILD_TYPE="${OPTARG}" ;;
    s) SWEEP=1 ;;
    c) CLEAN=1 ;;
    h) usage; exit 0 ;;
    \?) echo "Unknown option: -${OPTARG}" >&2; usage; exit 1 ;;
    :) echo "Option -${OPTARG} requires a value." >&2; usage; exit 1 ;;
  esac
done
shift $((OPTIND - 1))

if [[ $# -gt 0 && "$1" == "--" ]]; then
  shift
fi
EXTRA_ARGS=("$@")

if ! resolve_ascend_install_path; then
  echo "Cannot find ascend-toolkit. Please set ASCEND_INSTALL_PATH or pass -a <path>." >&2
  echo "Tried common paths under /usr/local/Ascend, /opt/Ascend, ${HOME}, and /workspace." >&2
  exit 1
fi

if [[ ! -d "${ASCEND_INSTALL_PATH}" ]]; then
  echo "ASCEND_INSTALL_PATH does not exist: ${ASCEND_INSTALL_PATH}" >&2
  exit 1
fi

if [[ ! -f "${ASCEND_INSTALL_PATH}/set_env.sh" ]]; then
  echo "Cannot find set_env.sh under ASCEND_INSTALL_PATH: ${ASCEND_INSTALL_PATH}" >&2
  exit 1
fi

source "${ASCEND_INSTALL_PATH}/set_env.sh"
echo "[INFO] ASCEND_INSTALL_PATH=${ASCEND_INSTALL_PATH}"

export ASCEND_INSTALL_PATH
export ASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}"
export SOC_VERSION

if [[ "${CLEAN}" == "1" ]]; then
  rm -rf "${BUILD_DIR}"
fi
mkdir -p "${BUILD_DIR}"
cd "${BUILD_DIR}"

cmake "${SCRIPT_DIR}" \
  -DCMAKE_BUILD_TYPE="${BUILD_TYPE}" \
  -DBUILD_ASCEND=ON \
  -DASCEND_CANN_PATH="${ASCEND_INSTALL_PATH}" \
  -DASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}" \
  -DSOC_VERSION="${SOC_VERSION}" \
  -DRUN_MODE="${RUN_MODE}"

cmake --build . -j

BIN="${BUILD_DIR}/rmsnorm_ascend_demo"
if [[ ! -x "${BIN}" ]]; then
  echo "Cannot find executable: ${BIN}" >&2
  exit 1
fi

CMD=("${BIN}" --device "${DEVICE_ID}" --m "${M}" --n "${N}" --tile-len "${TILE_LEN}" --block-dim "${BLOCK_DIM}" --epsilon "${EPSILON}" --version "${VERSION}" --warmup "${WARMUP}" --repeat "${REPEAT}" --seed "${SEED}")
if [[ "${SWEEP}" == "1" ]]; then
  CMD+=(--sweep)
fi
CMD+=("${EXTRA_ARGS[@]}")

echo "[RUN] ${CMD[*]}"
"${CMD[@]}"


In [ ]:
%%writefile Source/04.03/scripts/profile_msprof_template.sh

#!/usr/bin/env bash
set -euo pipefail

ROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)"
VERSION="${1:-pipeline}"
BIN="${ROOT}/build_ascend/rmsnorm_ascend_demo"
OUT_DIR="${ROOT}/profile_${VERSION}_$(date +%Y%m%d_%H%M%S)"

if [[ ! -x "${BIN}" ]]; then
  echo "Build the Ascend demo first with scripts/run_ascend.sh." >&2
  exit 1
fi

msprof \
  --application="${BIN} --device 0 --m 256 --n 4096 --tile-len 256 --block-dim 8 --epsilon 1e-5 --version ${VERSION} --warmup 5 --repeat 50 --seed 1234" \
  --output="${OUT_DIR}"


In [ ]:
!chmod +x Source/04.03/scripts/run_ascend.sh
!chmod +x Source/04.03/scripts/profile_msprof_template.sh
!find Source/04.03 -maxdepth 4 -type f | sort


### 5.10 实验参数与完整运行

| 参数 | 数值 | 参数意义 |
|---|---:|---|
| 输入规模M,N | 256,1024 | M为行数，N为每行归一化维度 |
| paddedN | 1024 | N按tileLen向上补齐后的行宽 |
| tileLen | 256 | 每个tile处理的连续FP32元素数量 |
| 每行tile数量 | 4 | `paddedN/tileLen` |
| blockDim | 8 | Kernel启动任务块数量，本实验对应8个AI Core |
| epsilon | 1e-5 | 加入均方值的数值稳定项 |
| warmup | 2 | 正式计时前的预运行次数 |
| repeat | 10 | 正式计时的重复次数 |
| seed | 1234 | 保证三种实验方案使用相同输入 |

下面先清理并编译一次工程，再依次运行三种实验方案。每种实验方案的完整终端输出同时写入`results`目录。


In [ ]:
%%bash
set -euo pipefail
cd Source/04.03
mkdir -p results

bash scripts/run_ascend.sh -c -M 256 -N 1024 -l 256 -b 8 -p 1e-5 -V basic -w 2 -r 10 -e 1234 \
  | tee results/basic.log
bash scripts/run_ascend.sh -c -M 256 -N 1024 -l 256 -b 8 -p 1e-5 -V fused -w 2 -r 10 -e 1234 \
  | tee results/fused.log
bash scripts/run_ascend.sh -c -M 256 -N 1024 -l 256 -b 8 -p 1e-5 -V pipeline -w 2 -r 10 -e 1234 \
  | tee results/pipeline.log


### 5.11 汇总运行结果

下面从三个日志中提取结果行，生成统一CSV并计算加速比。先检查`status`和`errors`，只有三种实验方案均为PASS时，性能比较才有意义。


In [ ]:
import csv
import re
from pathlib import Path
from IPython.display import Markdown, display

result_dir = Path("Source/04.03/results")
versions = ("basic", "fused", "pipeline")
rows = []

for version in versions:
    log_path = result_dir / f"{version}.log"
    if not log_path.exists():
        print("Missing log:", log_path)
        continue
    text = log_path.read_text(encoding="utf-8", errors="replace")
    data = None
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 14 and parts[6] == version:
            data = {
                "M": int(parts[0]),
                "N": int(parts[1]),
                "paddedN": int(parts[2]),
                "tileLen": int(parts[3]),
                "tiles": int(parts[4]),
                "cores": int(parts[5]),
                "version": parts[6],
                "kernel_us": float(parts[7]),
                "total_us": float(parts[8]),
                "GB/s": float(parts[9]),
                "max_abs": float(parts[10]),
                "max_rel": float(parts[11]),
                "errors": int(parts[12]),
                "status": parts[13],
            }
            break
    if data is None:
        print("No result row found in:", log_path)
        continue
    moved = re.search(r"estimatedMovedBytes=(\d+)", text)
    data["estimatedMovedBytes"] = int(moved.group(1)) if moved else None
    rows.append(data)

if rows:
    csv_path = result_dir / "rmsnorm_results.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    headers = ["version", "kernel_us", "total_us", "GB/s", "max_abs", "max_rel", "errors", "status"]
    table = ["|" + "|".join(headers) + "|", "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        table.append("|" + "|".join(str(row[name]) for name in headers) + "|")
    display(Markdown("\n".join(table)))

    by_version = {row["version"]: row for row in rows}
    if all(name in by_version for name in versions):
        basic = by_version["basic"]["kernel_us"]
        fused = by_version["fused"]["kernel_us"]
        pipeline = by_version["pipeline"]["kernel_us"]
        print(f"basic -> fused speedup: {basic / fused:.2f}x")
        print(f"fused -> pipeline speedup: {fused / pipeline:.2f}x")
        print(f"basic -> pipeline speedup: {basic / pipeline:.2f}x")
        print("All versions PASS:", all(row["status"] == "PASS" and row["errors"] == 0 for row in rows))
    print("CSV saved to:", csv_path)


### 5.12 结果判读与性能分析

按以下顺序分析结果：

1. **正确性**：三种实验方案应满足`errors=0`且`status=PASS`；最大绝对误差和最大相对误差反映浮点归约顺序带来的数值差异。
2. **基线实验方案与单Kernel连续计算实验方案**：比较Kernel数量和`estimatedMovedBytes`。性能提升同时来自平方中间结果GM访问减少、Kernel融合及Vector归约组织。
3. **单Kernel连续计算实验方案与双缓冲流水实验方案**：两者计算步骤、两遍读取X、gamma读取、tileLen和估算GM访问量一致。主要观察双缓冲队列是否降低`kernel_us`和`total_us`。
4. **规模效应**：每行只有少量tile时，队列管理与流水启动/排空可能抵消重叠收益；tile数量增加后，流水稳定阶段更长。
5. **计时波动**：不同批次结果可能受设备负载、频率、调度和缓存状态影响，应结合预热、重复测量和多次实验判断趋势。

`GB/s`由估算搬运量和总耗时计算，只用于本实验方案间的相对比较。


### 5.13 Profiling观察

先完成正常构建，再在终端中分别采集单Kernel连续计算实验方案（命令参数`fused`）和双缓冲流水实验方案（命令参数`pipeline`）：

```bash
cd ~/Source/04.03
bash scripts/profile_msprof_template.sh fused
bash scripts/profile_msprof_template.sh pipeline
```

分析时重点比较Kernel执行时间，并观察MTE2数据搬入、Vector计算和MTE3写回在时间线上的分布。双缓冲流水实验方案出现阶段交错才说明双缓冲形成了实际重叠；仅使用TQue或两个缓冲区并不能单独证明性能一定提高。


### 5.14 扩展实验

#### 行宽N对比：观察每行tile数量

保持`M=256`、`tileLen=256`和`blockDim=8`不变，只改变N，可以单独观察每行tile数量对流水启动、稳定执行和排空开销的影响。每个N都完整运行基线实验方案、单Kernel连续计算实验方案和双缓冲流水实验方案。

```bash
cd ~/Source/04.03
mkdir -p results/scale_n
# N=1024：每行4个tile
bash scripts/run_ascend.sh -c -M 256 -N 1024 -l 256 -b 8 -p 1e-5 -V basic -w 5 -r 50 -e 1234 \
  | tee results/scale_n/M256_N1024_basic.log
bash scripts/run_ascend.sh -c -M 256 -N 1024 -l 256 -b 8 -p 1e-5 -V fused -w 5 -r 50 -e 1234 \
  | tee results/scale_n/M256_N1024_fused.log
bash scripts/run_ascend.sh -c -M 256 -N 1024 -l 256 -b 8 -p 1e-5 -V pipeline -w 5 -r 50 -e 1234 \
  | tee results/scale_n/M256_N1024_pipeline.log

# N=4096：每行16个tile
bash scripts/run_ascend.sh -c -M 256 -N 4096 -l 256 -b 8 -p 1e-5 -V basic -w 5 -r 50 -e 1234 \
  | tee results/scale_n/M256_N4096_basic.log
bash scripts/run_ascend.sh -c -M 256 -N 4096 -l 256 -b 8 -p 1e-5 -V fused -w 5 -r 50 -e 1234 \
  | tee results/scale_n/M256_N4096_fused.log
bash scripts/run_ascend.sh -c -M 256 -N 4096 -l 256 -b 8 -p 1e-5 -V pipeline -w 5 -r 50 -e 1234 \
  | tee results/scale_n/M256_N4096_pipeline.log

# N=8192：每行32个tile
bash scripts/run_ascend.sh -c -M 256 -N 8192 -l 256 -b 8 -p 1e-5 -V basic -w 5 -r 50 -e 1234 \
  | tee results/scale_n/M256_N8192_basic.log
bash scripts/run_ascend.sh -c -M 256 -N 8192 -l 256 -b 8 -p 1e-5 -V fused -w 5 -r 50 -e 1234 \
  | tee results/scale_n/M256_N8192_fused.log
bash scripts/run_ascend.sh -c -M 256 -N 8192 -l 256 -b 8 -p 1e-5 -V pipeline -w 5 -r 50 -e 1234 \
  | tee results/scale_n/M256_N8192_pipeline.log
```

#### 行数M对比：观察总工作量

保持`N=4096`不变，只改变M，可以观察输入行数和每个AI Core处理行数对总体性能的影响。此时每行始终为16个tile，不改变单行内部的流水长度。

```bash
mkdir -p results/scale_m
# M=64
bash scripts/run_ascend.sh -c -M 64 -N 4096 -l 256 -b 8 -p 1e-5 -V basic -w 5 -r 50 -e 1234 \
  | tee results/scale_m/M64_N4096_basic.log
bash scripts/run_ascend.sh -c -M 64 -N 4096 -l 256 -b 8 -p 1e-5 -V fused -w 5 -r 50 -e 1234 \
  | tee results/scale_m/M64_N4096_fused.log
bash scripts/run_ascend.sh -c -M 64 -N 4096 -l 256 -b 8 -p 1e-5 -V pipeline -w 5 -r 50 -e 1234 \
  | tee results/scale_m/M64_N4096_pipeline.log

# M=256
bash scripts/run_ascend.sh -c -M 256 -N 4096 -l 256 -b 8 -p 1e-5 -V basic -w 5 -r 50 -e 1234 \
  | tee results/scale_m/M256_N4096_basic.log
bash scripts/run_ascend.sh -c -M 256 -N 4096 -l 256 -b 8 -p 1e-5 -V fused -w 5 -r 50 -e 1234 \
  | tee results/scale_m/M256_N4096_fused.log
bash scripts/run_ascend.sh -c -M 256 -N 4096 -l 256 -b 8 -p 1e-5 -V pipeline -w 5 -r 50 -e 1234 \
  | tee results/scale_m/M256_N4096_pipeline.log

# M=512
bash scripts/run_ascend.sh -c -M 512 -N 4096 -l 256 -b 8 -p 1e-5 -V basic -w 5 -r 50 -e 1234 \
  | tee results/scale_m/M512_N4096_basic.log
bash scripts/run_ascend.sh -c -M 512 -N 4096 -l 256 -b 8 -p 1e-5 -V fused -w 5 -r 50 -e 1234 \
  | tee results/scale_m/M512_N4096_fused.log
bash scripts/run_ascend.sh -c -M 512 -N 4096 -l 256 -b 8 -p 1e-5 -V pipeline -w 5 -r 50 -e 1234 \
  | tee results/scale_m/M512_N4096_pipeline.log
```

#### tileLen对比

固定`M=256,N=4096`，使用`sweep`分别扫描单Kernel连续计算实验方案和双缓冲流水实验方案支持的tileLen。这组实验专门对比两种实验方案，因此不再重复运行基线实验方案：

```bash
mkdir -p results/tile_sweep
bash scripts/run_ascend.sh -c -M 256 -N 4096 -b 8 -p 1e-5 -V fused -w 5 -r 50 -e 1234 -s \
  | tee results/tile_sweep/fused.log
bash scripts/run_ascend.sh -c -M 256 -N 4096 -b 8 -p 1e-5 -V pipeline -w 5 -r 50 -e 1234 -s \
  | tee results/tile_sweep/pipeline.log
```

分析时结合`tileLen`、每行tile数量和片上缓冲占用解释结果。tile过小会增加循环、同步和队列操作；tile过大则增加UB占用，并可能限制双缓冲可用容量。


---
## 6. 实验总结

本实验实现了RMSNorm的基线实验方案、单Kernel连续计算实验方案和双缓冲流水实验方案。基线实验方案通过两个Kernel保留平方中间结果；单Kernel连续计算实验方案把整行归约和输出计算融合到一个Kernel；双缓冲流水实验方案保持与单Kernel连续计算实验方案相同的数学过程和主要GM访问量，使用深度为2的TQue组织相邻tile。

RMSNorm的整行归约依赖决定了流水必须保留“先完成整行平方和，再计算归一化因子，最后计算输出”的顺序。双缓冲只能在两遍计算各自的tile序列内部建立搬入、Vector计算和写回的阶段化推进。

正确性校验、预热、重复测量、有效带宽估算和Profiling共同构成性能分析依据。实验结论应限定在具体M、N、tileLen和blockDim配置下，并通过规模与tileLen扩展实验判断流水收益是否稳定。
